## 1 setup

In [ ]:
%pip install -q pandas pyarrow numpy tqdm

In [ ]:
from pathlib import Path
import json
import math
import warnings #gestire warnings
import numpy as np 
import pandas as pd
import pyarrow.parquet as pq 
from tqdm.auto import tqdm #barra di progresso

warnings.filterwarnings("ignore", category=FutureWarning) #nasconde avvisi per avere output meno rumoroso
#impostazioni di pandas
pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",140)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")  

## 2 percorsi e configurazione scelta nel notebook 02

In [3]:
CATEGORY="AmazonLike_v1" #dataset scelto
DATASET_NAME="test_dataset2" #nome tecnico usato per le cartelle
SELECTED_CONFIG="user3_item8" #configurazione scelta nel notebook02
POSITIVE_RATING_THRESHOLDS=4 #rating minimo per considerare una interazione positiva
MIN_USER_INTERACTIONS=3
MIN_ITEM_INTERACTIONS=8
N_NEG=100 #quanti item negativi campionare per ogni esempio di validation/test
NEGATIVE_SAMPLING_MODE="mixed_category" #mixed_category usa metà negativi della stessa categoria e metà globali
SAME_CATEGORY_NEGATIVE_RATIO=0.50 #quota teorica dei negativi presi dalla stessa source_category
RANDOM_STATE=42 #seed casuale per rendere il negative sampling riproducibile


In [4]:
current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        directory
        for directory in (current_dir, *current_dir.parents)
        if (directory / "requirements.txt").is_file() and (directory / "prototype").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Avvia Jupyter dalla cartella del repository o da una sua sottocartella.")
SOURCE_DIR=PROJECT_ROOT/"data"/"processed"/f"{DATASET_NAME}_intersection" #output dei precedenti notebook
ANALYSIS_DIR=SOURCE_DIR/"analysis" #riepiloghi notebook 2
OUTPUT_DIR=PROJECT_ROOT/"data"/"processed"/f"{DATASET_NAME}_recommender" #output di questo notebook03
OUTPUT_DIR.mkdir(parents=True,exist_ok=True) #crea cartella output se non esiste

CANDIDATE_INTERACTIONS_PATH=SOURCE_DIR/f"candidate_interactions_{SELECTED_CONFIG}.parquet" #interazioni candidate
CANDIDATE_ITEMS_PATH=SOURCE_DIR/f"candidate_items_{SELECTED_CONFIG}.parquet" #prodotti candidati
CANDIDATE_QA_PATH=SOURCE_DIR/f"candidate_qa_{SELECTED_CONFIG}.parquet" #qa candidati
NOTEBOOK02_CONFIG_PATH=ANALYSIS_DIR/"notebook02_selected_filter_config.json" #riepilogo config scelta

print("Project root:",PROJECT_ROOT)
print("Source dir:",SOURCE_DIR)
print("Output dir:",OUTPUT_DIR)


Project root: .
Source dir: ./data/processed/test_dataset2_intersection
Output dir: ./data/processed/test_dataset2_recommender


## 3 caricamento dei dati candidati

In [5]:
#carichiamo dati
interactions_df=pd.read_parquet(CANDIDATE_INTERACTIONS_PATH)
items_df=pd.read_parquet(CANDIDATE_ITEMS_PATH)
qa_df=pd.read_parquet(CANDIDATE_QA_PATH)
notebook02_Config=json.loads(NOTEBOOK02_CONFIG_PATH.read_text(encoding="utf-8"))
#stampiamo dimensioni tabelle
print("interactions_df:",interactions_df.shape)
print("items_df:",items_df.shape)
print("qa_df:",qa_df.shape)
display(pd.DataFrame([notebook02_Config])) #riepilogo dati dentro configurazione scelta

interactions_df: (1494991, 12)
items_df: (22664, 16)
qa_df: (193922, 12)


,selected_config_name,min_user_interactions,min_item_interactions,num_users,num_items,num_interactions,density_pct,sparsity_pct,category,dataset_name
0,user3_item8,3,8,336898,22664,1494991,0.019580,99.980420,AmazonLike_v1,test_dataset2


## 4 prepariamo timestamp

In [6]:
TIME_COL="datetime" #colonna temporale usata nello split

if TIME_COL in interactions_df.columns: 
    interactions_df[TIME_COL]=pd.to_datetime( #convertiamola in datetime pandas 
        interactions_df[TIME_COL], 
        errors="coerce", #trasforma errori in NaT
    )
elif "timestamp" in interactions_df.columns: #se datetime non c'è, usa timestamp
    interactions_df[TIME_COL]=pd.to_datetime(
        interactions_df["timestamp"],
        unit="ms",
        errors="coerce",
    )
else:
    raise ValueError("mancano sia timestamp sia datetime")

invalid_datetime_rows=int(interactions_df[TIME_COL].isna().sum()) #conta righe senza datetime valido
if invalid_datetime_rows>0:
    print("righe rimosse senza datetime valido:", invalid_datetime_rows)
    interactions_df=interactions_df.dropna(subset=[TIME_COL]).copy() #rimuove righe


print("Colonna temporale usata per lo split:",TIME_COL)
print("Data minima:",interactions_df[TIME_COL].min())
print("Data massima:",interactions_df[TIME_COL].max()) 

preview_c=["timestamp",TIME_COL] if "timestamp" in interactions_df.columns else [TIME_COL]
display(interactions_df[preview_c].head())

Colonna temporale usata per lo split: datetime
Data minima: 1999-08-21 22:04:38
Data massima: 2023-07-20 14:41:36.407000


,timestamp,datetime
0,1275251437000,2010-05-30 20:30:37
1,1258206280000,2009-11-14 13:44:40
2,1275247369000,2010-05-30 19:22:49
3,1311718723000,2011-07-26 22:18:43
4,1344043262000,2012-08-04 01:21:02


## 5 definizione target: feedback positivo
consideriamo coppie come rilevanti se rating è almeno 4

In [7]:
rating_dist=( 
    #distribuzione dei rating nel dataset candidato, la rifacciamo perchè potrebbe essere
    #diverso dopo aver filtrato con la configurazione scelta
    interactions_df["rating"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="count")
)
rating_dist["percentage"]=rating_dist["count"]/len(interactions_df)*100
display(rating_dist)

,rating,count,percentage
0,1.000000,113710,7.606066
1,2.000000,73640,4.925782
2,3.000000,121823,8.148745
3,4.000000,263164,17.603049
4,5.000000,922654,61.716358


In [8]:
positive_interactions_df=interactions_df[interactions_df["rating"]>= POSITIVE_RATING_THRESHOLDS].copy() #tiene solo rating pos
positive_interactions_df["interaction_value"]=1 #crea colonna binaria cosi capiamo i rating positivi
target_stats={
    "original_interaction":int(len(interactions_df)), #numero di interazioni candidate originali
    "positive_interaction":int(len(positive_interactions_df)), #n int con rating >=soglia
    "removed_interaction":int(len(interactions_df)-len(positive_interactions_df)), #num int escluse
    "positive_int_percentage":float(len(positive_interactions_df)/len(interactions_df)*100), # % interazioni positive
    "original_users_num":int(interactions_df["user_id"].nunique()), #utenti prima
    "positive_users_num":int(positive_interactions_df["user_id"].nunique()), #utenti con almeno un rating positivo
    "original_items":int(interactions_df["parent_asin"].nunique()), #items prima
    "positive_items":int(positive_interactions_df["parent_asin"].nunique()), #prodotti con almeno un rating positivo
}

target_stats_df=pd.DataFrame.from_dict(target_stats,orient="index",columns=["value"]) #tabella
display(target_stats_df)

,value
original_interaction,"1,494,991.000000"
positive_interaction,"1,185,818.000000"
removed_interaction,"309,173.000000"
positive_int_percentage,79.319407
original_users_num,"336,898.000000"
positive_users_num,"329,281.000000"
original_items,"22,664.000000"
positive_items,"22,660.000000"


## 6 Rifiltraggio iterativo dopo la scelta del target
Dopo aver tenuto solo rating>=4 dobbiamo ricontrollare che tutti gli utenti abbiamo >=3 interazioni e tutti i prodotti >=8 interazioni

In [9]:
def iter_filter(df, min_user_interactions,min_item_interactions,user_col="user_id",item_col="parent_asin",max_iter=50):
    #funzione per filtro iterativo
    current_df=df.copy()
    history_rows=[]
    for iteration in range(1,max_iter+1):
        #numeri prima del filtro
        rows_before=len(current_df)
        users_before=current_df[user_col].nunique()
        items_before=current_df[item_col].nunique()
        if len(current_df)==0: #se vuoto il dataframe
            break
        item_num=current_df[item_col].value_counts() #conta interazioni positive per prodotto
        valid_items=item_num[item_num>=min_item_interactions].index #seleziona prodotti sopra la soglia
        current_df=current_df[current_df[item_col].isin(valid_items)].copy() #mantiene solo prodotti validi
        user_counts=current_df[user_col].value_counts() #conta int positiv per utente
        valid_users=user_counts[user_counts>=min_user_interactions].index #seleziona utenti sopra soglia
        current_df=current_df[current_df[user_col].isin(valid_users)].copy() #tiene solo utenti validi
        #contiamo righe dopo filtro
        rows_after=len(current_df)
        users_after=current_df[user_col].nunique()
        items_after=current_df[item_col].nunique()
        history_rows.append({ #statistiche dell'iterazione
            "iteration":iteration,
            "rows_before":rows_before,
            "rows_after":rows_after,
            "rows_removed":rows_before-rows_after,
            "users_before":users_before,
            "users_after":users_after,
            "users_removed":users_before-users_after,
            "items_before":items_before,
            "items_after":items_after,
            "items_removed":items_before-items_after,
        })
        if rows_after==rows_before:
            break

    history_df=pd.DataFrame(history_rows)
    return current_df.reset_index(drop=True), history_df


In [10]:

filtered_positive_df, filter_history_df=iter_filter(
    positive_interactions_df,
    min_user_interactions=MIN_USER_INTERACTIONS,
    min_item_interactions=MIN_ITEM_INTERACTIONS,
    user_col="user_id",
    item_col="parent_asin",
    max_iter=50,
)
print("interazioni positive dopo filtraggio:", len(filtered_positive_df))
print("utenti dopo filtraggio:",filtered_positive_df["user_id"].nunique())
print("prodotti dopo filtraggio:",filtered_positive_df["parent_asin"].nunique())
display(filter_history_df)

interazioni positive dopo filtraggio: 992235
utenti dopo filtraggio: 229426
prodotti dopo filtraggio: 19000


,iteration,rows_before,rows_after,rows_removed,users_before,users_after,users_removed,items_before,items_after,items_removed
0,1,1185818,1007856,177962,329281,232743,96538,22660,20395,2265
1,2,1007856,992965,14891,232743,229572,3171,20395,19063,1332
2,3,992965,992301,664,229572,229438,134,19063,19006,57
3,4,992301,992244,57,229438,229427,11,19006,19001,5
4,5,992244,992235,9,229427,229426,1,19001,19000,1
5,6,992235,992235,0,229426,229426,0,19000,19000,0


In [11]:
final_user_counts=filtered_positive_df["user_id"].value_counts()
final_item_counts=filtered_positive_df["parent_asin"].value_counts()

post_filter_check={ #dizionario per controlli
    "min_user_int":int(final_user_counts.min())if len(final_user_counts)else 0,
    "min_item_int":int(final_item_counts.min())if len(final_item_counts)else 0,
    "users_with_3int":int((final_user_counts>=MIN_USER_INTERACTIONS).sum()),
    "items_with_8int":int((final_item_counts>=MIN_ITEM_INTERACTIONS).sum()),
    "all_thresholds_ok":bool(
        len(final_user_counts)>0
        and len(final_item_counts)>0
        and final_user_counts.min()>=MIN_USER_INTERACTIONS
        and final_item_counts.min()>=MIN_ITEM_INTERACTIONS
    ),
}

display(pd.DataFrame.from_dict(post_filter_check,orient="index",columns=["value"]))
if not post_filter_check["all_thresholds_ok"]:
    raise ValueError("Dopo filtro non tutte le soglie minime sono rispettate")

,value
min_user_int,3
min_item_int,8
users_with_3int,229426
items_with_8int,19000
all_thresholds_ok,True


## 7 filtraggio prodotti e qa

In [12]:
final_products=set(filtered_positive_df["parent_asin"].unique()) #crea insieme prodotti finali
final_items_df=items_df[items_df["parent_asin"].isin(final_products)].copy() #filtra i metadati sui prodotti finali
final_qa_df=qa_df[qa_df["parent_asin"].isin(final_products)].copy() #filtra qa

print("final_items_df:",final_items_df.shape)
print("final_qa_df:",final_qa_df.shape)
print("Prodotti finali con metadati:",final_items_df["parent_asin"].nunique())
print("Prodotti finali con QA:",final_qa_df["parent_asin"].nunique())

final_items_df: (19000, 16)
final_qa_df: (166191, 12)
Prodotti finali con metadati: 19000
Prodotti finali con QA: 19000


In [13]:
#controlliamo la distribuzione finale per categoria
#utile per capire se il dataset multi-categoria resta bilanciato dopo il target rating>=4
final_category_stats_df=(
    filtered_positive_df.groupby("source_category") #raggruppa per riga
    .agg(
        positive_interactions=("rating","size"), #int positive
        users=("user_id","nunique"), #n utenti con int positive
        products=("parent_asin","nunique"), #n prod con int positive
        avg_rating=("rating","mean"), #rating medio delle sole interazioni positive
    )
    .reset_index()
    .sort_values("positive_interactions",ascending=False)
)
display(final_category_stats_df)


,source_category,positive_interactions,users,products,avg_rating
2,Electronics,706050,219145,13686,4.785883
1,Cell Phones and Accessories,121963,79380,2573,4.785255
3,Office Products,85274,62776,1618,4.794955
5,Video Games,49983,33092,481,4.821279
0,Appliances,15088,13440,312,4.843584
4,Software,13877,12219,330,4.733876


## 8 creiamo indici numerici

In [14]:
user_mapping_df=pd.DataFrame({"user_id":sorted(filtered_positive_df["user_id"].unique())})#tab utenti ordinata
user_mapping_df["user_idx"]=np.arange(len(user_mapping_df),dtype=np.int64)

item_mapping_df=pd.DataFrame({"parent_asin":sorted(filtered_positive_df["parent_asin"].unique())})#tab prodotti ordinata
item_mapping_df["item_idx"]=np.arange(len(item_mapping_df),dtype=np.int64)

indexed_interactions_df=filtered_positive_df.merge(user_mapping_df,on="user_id",how="left")
indexed_interactions_df=indexed_interactions_df.merge(item_mapping_df,on="parent_asin",how="left")

final_items_df=final_items_df.merge(item_mapping_df,on="parent_asin",how="left")
final_qa_df=final_qa_df.merge(item_mapping_df,on="parent_asin",how="left")

print("Utenti mappati:",len(user_mapping_df))
print("Prodotti mappati:",len(item_mapping_df))
display(user_mapping_df.head())
display(item_mapping_df.head())

Utenti mappati: 229426
Prodotti mappati: 19000


,user_id,user_idx
0,AE22236AFRRSMQIKGG7TPTB75QEA,0
1,AE222H3FGXWLHRFUMGMS2RR57NDQ,1
2,AE223LQXDVA2DHNB7XKAPILBS2DQ,2
3,AE223NW3QAU2KTAY5MAEU6VPBAUA,3
4,AE223SBXB2PFTMOITN4V53TLSFSA,4


,parent_asin,item_idx
0,0615179088,0
1,0792283090,1
2,0972683275,2
3,1400501466,3
4,1400532655,4


In [15]:
#in caso di errori dopo il merge, a causa di duplicati o NaN riportiamo a interi, teoricamente è tutto già stato controllato, ma per sicurezza lo facciamo
indexed_interactions_df["user_idx"]=indexed_interactions_df["user_idx"].astype(np.int64)
indexed_interactions_df["item_idx"]=indexed_interactions_df["item_idx"].astype(np.int64)
final_items_df["item_idx"]=final_items_df["item_idx"].astype(np.int64)
if "item_idx" in final_qa_df.columns:
    final_qa_df["item_idx"]=final_qa_df["item_idx"].astype(np.int64)

## 9 split temporale train/val/test
ultima in test, penultima in val, il resto in training

In [16]:
indexed_interactions_df["interaction_id"]=np.arange(len(indexed_interactions_df),dtype=np.int64) #crea id interazione
indexed_interactions_df=indexed_interactions_df.sort_values(
    ["user_idx",TIME_COL,"interaction_id"], #ordina
    ascending=[True,True,True],
).reset_index(drop=True) #ricrea l'indice

indexed_interactions_df["n_userhistory"]=indexed_interactions_df.groupby("user_idx").cumcount() #posizione temporale fra le varie interazioni dell'utente
indexed_interactions_df["user_history_length"]=indexed_interactions_df.groupby("user_idx")["item_idx"].transform("size") #num tot int utente
#ultima in test, penultima in val, altre in training
indexed_interactions_df["split"]="train"
validation_mask=indexed_interactions_df["n_userhistory"]==indexed_interactions_df["user_history_length"]-2
test_mask=indexed_interactions_df["n_userhistory"]==indexed_interactions_df["user_history_length"]-1

indexed_interactions_df.loc[validation_mask,"split"]="validation"
indexed_interactions_df.loc[test_mask,"split"]="test"
#estraiamo i set
train_df=indexed_interactions_df[indexed_interactions_df["split"]=="train"].copy()
validation_df=indexed_interactions_df[indexed_interactions_df["split"]=="validation"].copy()
test_df=indexed_interactions_df[indexed_interactions_df["split"]=="test"].copy()

print("Train interactions:",len(train_df))
print("validation interactions:",len(validation_df))
print("test interactions:",len(test_df))

Train interactions: 533383
validation interactions: 229426
test interactions: 229426


In [17]:
split_summary_df=(
    indexed_interactions_df.groupby("split")
    .agg(
        num_rows=("interaction_id","size"),
        num_users=("user_idx","nunique"),
        num_items=("item_idx","nunique"),
        min_time=(TIME_COL,"min"),
        max_time=(TIME_COL,"max"), 
    )
    .reset_index()
)
display(split_summary_df)

,split,num_rows,num_users,num_items,min_time,max_time
0,test,229426,229426,17802,2000-11-26 20:22:16,2023-07-20 14:41:36.407
1,train,533383,229426,18996,1999-08-21 22:04:38,2023-07-20 14:39:32.308
2,validation,229426,229426,18405,2000-05-17 19:52:55,2023-07-20 14:40:07.816


## 10 controllo cold start sugli item
non vogliamo prodotti nel val e test mai visti nel training

In [18]:
#insieme prodotti dei vari split
train_items=set(train_df["item_idx"].unique())
validation_items=set(validation_df["item_idx"].unique())
test_items=set(test_df["item_idx"].unique())

validation_cold_items=validation_items-train_items #prod val assenti dal train
test_cold_items=test_items-train_items #prod test assenti dal train

#val e test migliorati, per uso di collaborative filtering
validation_eval_df=validation_df[validation_df["item_idx"].isin(train_items)].copy()
test_eval_df=test_df[test_df["item_idx"].isin(train_items)].copy()

cold_start_stats={ #statistiche cold start
    "train_items":int(len(train_items)),
    "validation_items":int(len(validation_items)),
    "test_items":int(len(test_items)),
    "validation_cold_items":int(len(validation_cold_items)),
    "test_cold_items":int(len(test_cold_items)),
    "validation_rows_original":int(len(validation_df)),
    "validation_rows_eval":int(len(validation_eval_df)),
    "test_rows_original":int(len(test_df)),
    "test_rows_eval":int(len(test_eval_df)),
}

display(pd.DataFrame.from_dict(cold_start_stats, orient="index", columns=["value"]))

,value
train_items,18996
validation_items,18405
test_items,17802
validation_cold_items,4
test_cold_items,4
validation_rows_original,229426
validation_rows_eval,229415
test_rows_original,229426
test_rows_eval,229395


## 11 negative sampling
per confrontare l'item positivo reale, usiamo N_NEG prodotti del train escludendo prodotti positivi per quell'utente

In [19]:
user_seen_items=( #elenco item positivi per ogni utente
    indexed_interactions_df.groupby("user_idx")["item_idx"] #raggruppa int positive per utente e prende solo colonna item_idx
    .apply(lambda values:set(values.astype(int))) #converte la lista di item in insieme
    .to_dict() #converte in dizionario
)
candidate_item_idx=np.array(sorted(train_items),dtype=np.int64) #usa come candidati neg solo item presenti nel train

item_category_lookup=final_items_df.set_index("item_idx")["source_category"].to_dict() #trasformi in dizionario, id e categoria
train_items_with_category_df=final_items_df[final_items_df["item_idx"].isin(train_items)].copy() #tieni solo quelli presenti nel train
category_candidate_items={ #costruisci dizionario, per ogni categoria, raccogli itemidx del train
    str(category):np.array(sorted(group["item_idx"].astype(int).unique()),dtype=np.int64)
    for category,group in train_items_with_category_df.groupby("source_category")
}

print("Numero item candidati per negative samples:",len(candidate_item_idx))
print("Categorie candidate per negative samples:",len(category_candidate_items))


Numero item candidati per negative samples: 18996
Categorie candidate per negative samples: 6


In [20]:
#campionamento negativi, cioè prodotti che l'utente non ha già valutato positivamente
#con mixed_category proviamo a prendere una parte dei negativi dalla stessa categoria del positivo

#poolitems è insieme dei prodotti da cui pescare, positiveitem è nostro prodotto da testare, seenitems è prodotti che utente ha già valutato positivamente
#needed è quanti neg vuoi avere, rng generatore casuale riproducibile, negativeitems è lista negativi scelti, negativeset è uguale ma set quindi veloce per controllo se items è stato già scelto
def sample_from_pool(pool_items,positive_item_idx,seen_items,needed,rng,negative_items,negative_set):
    if needed<=0 or len(pool_items)==0:
        return negative_items
    pool_items=np.asarray(pool_items,dtype=np.int64)
    max_rounds=20
    sample_size=min(len(pool_items),max(needed*4,needed+len(seen_items))) #quanti prodotti pescare a ogni tentativo
    for _ in range(max_rounds):
        sampled_items=rng.choice(pool_items,size=sample_size,replace=False)
        for sampled_item in sampled_items:
            sampled_item=int(sampled_item)
            if sampled_item==int(positive_item_idx): #non deve essere il prodotto corrente
                continue
            if sampled_item in seen_items: #non deve essere già visto positivamente dall'utente
                continue
            if sampled_item in negative_set: #non deve essere già stato scelto come negativo
                continue
            negative_items.append(sampled_item)
            negative_set.add(sampled_item)
            if len(negative_items)>=needed:
                return negative_items
    return negative_items

#useridx utente per cui stai generando i negativi
#positiveitemidx id del item pos corrente
#positivecategory categoria del prodotto positivo, se disponibile
#candidateitems insieme item globali da cui puoi pescare negativi
#seenitemsbyuser dizionario per ogni utente, tutti gli item positivi osservati
#n_neg=100
#rng generatore casuale
def sample_neg_items(user_idx,positive_item_idx,positive_category,candidate_items,seen_items_byuser,n_neg,rng):
    seen_items=seen_items_byuser.get(int(user_idx),set()) #recupera gli item pos già osservati dell'utente
    negative_items=[]
    negative_set=set()
    #prendo una parte dei negativi dalla stessa categoria
    if NEGATIVE_SAMPLING_MODE=="mixed_category" and positive_category in category_candidate_items:
        same_category_target=int(round(n_neg*SAME_CATEGORY_NEGATIVE_RATIO)) #calcolo quanti prenderne dalla stessa categoria
        same_category_pool=category_candidate_items.get(str(positive_category),candidate_items) #prodotti candidati della stessa categoria
        negative_items=sample_from_pool( #prendi gli items negativi
            same_category_pool,
            positive_item_idx,
            seen_items,
            same_category_target,
            rng,
            negative_items,
            negative_set,
        )

    negative_items=sample_from_pool( #poi completiamo con negativi globali, può essere che alcuni siano comunque dalla categoria di prima
        candidate_items,
        positive_item_idx,
        seen_items,
        n_neg,
        rng,
        negative_items,
        negative_set,
    )

    if len(negative_items)<n_neg: #se non abbiamo abbastanza negativi, entriamo nel fallback
        available_items=[int(item) for item in candidate_items if int(item) not in seen_items and int(item)!=int(positive_item_idx)]
        available_items=[item for item in available_items if item not in negative_set]
        remaining_needed=n_neg-len(negative_items)
        if len(available_items)<remaining_needed:
            raise ValueError(f"Non ci sono abbastanza negativi per user_idx={user_idx}")
        extra_items=rng.choice(np.array(available_items,dtype=np.int64),size=remaining_needed,replace=False)
        negative_items.extend([int(item)for item in extra_items])

    return negative_items

In [21]:
#evaldf il dataframe da valutare
#splitname il nome dello split
def build_negative_samples(eval_df,split_name,n_neg=N_NEG,seed=RANDOM_STATE): #costruzione negative samples
    rng=np.random.default_rng(seed) #crea generatore casuale, ma riproducibile
    sample_rows=[]
    ordered_eval_df=eval_df.sort_values(["user_idx",TIME_COL,"interaction_id"]).reset_index(drop=True) #ordiniamo le righe da valutare
    for row in tqdm(ordered_eval_df.itertuples(index=False),total=len(ordered_eval_df),desc=f"Negative sampling {split_name}"):
        positive_category=getattr(row,"source_category",None)
        negatives=sample_neg_items( #campiona item negativi per la riga corrente
            user_idx=row.user_idx,
            positive_item_idx=row.item_idx,
            positive_category=positive_category,
            candidate_items=candidate_item_idx,
            seen_items_byuser=user_seen_items,
            n_neg=n_neg,
            rng=rng,
        )
        sample_rows.append({ #salva una riga di negative sampling
            "split":split_name,
            "user_idx":int(row.user_idx),
            "user_id":row.user_id,
            "positive_item_idx":int(row.item_idx),
            "positive_parent_asin":row.parent_asin,
            "positive_source_category":positive_category,
            "negative_item_idxs":negatives,
            "num_negatives":int(len(negatives)),
        })
    return pd.DataFrame(sample_rows)

In [22]:
validation_negative_samples_df=build_negative_samples(validation_eval_df,"validation",seed=RANDOM_STATE) #crea negativi validation
test_negatives_samples_df=build_negative_samples(test_eval_df,"test",seed=RANDOM_STATE+1) #negativi test ma seed diverso

print("validation_negative_samples_df:",validation_negative_samples_df.shape)
print("test_negative_samples_df:",test_negatives_samples_df.shape)
display(validation_negative_samples_df.head())
display(test_negatives_samples_df.head())


validation_negative_samples_df: (229415, 8)
test_negative_samples_df: (229395, 8)


,split,user_idx,user_id,positive_item_idx,positive_parent_asin,positive_source_category,negative_item_idxs,num_negatives
0,validation,0,AE22236AFRRSMQIKGG7TPTB75QEA,6514,B003ZDO2LM,Electronics,"[6528, 9018, 7947, 1663, 1667, 6210, 9065, 11692, 4199, 10787, 10123, 17156, 9922, 14360, 13938, 6848, 14205, 12535, 14361, 4048, 15656,...",100
1,validation,1,AE222H3FGXWLHRFUMGMS2RR57NDQ,8620,B0054JJ0QW,Electronics,"[9500, 3472, 18447, 108, 2821, 11216, 5422, 2740, 7983, 18783, 6651, 16805, 13464, 445, 11608, 3548, 17867, 9596, 8431, 5109, 14952, 125...",100
2,validation,2,AE223LQXDVA2DHNB7XKAPILBS2DQ,2409,B000Z7AKGC,Electronics,"[6300, 1223, 3559, 15893, 6430, 8971, 4367, 1665, 15335, 18032, 5319, 17590, 1090, 17192, 10396, 7313, 13734, 3807, 13583, 18017, 2929, ...",100
3,validation,3,AE223NW3QAU2KTAY5MAEU6VPBAUA,8210,B004VRQNS6,Electronics,"[13975, 14699, 1135, 2084, 6032, 8377, 8803, 7208, 1591, 16433, 18452, 11349, 11602, 17065, 6775, 1393, 1880, 8353, 6736, 17544, 3058, 4...",100
4,validation,4,AE223SBXB2PFTMOITN4V53TLSFSA,18285,B00HJ1NQLS,Electronics,"[13666, 18430, 9327, 4697, 7026, 4014, 16181, 9430, 847, 10157, 14688, 5436, 18876, 3823, 18658, 9035, 1630, 9598, 3743, 1566, 7692, 445...",100


,split,user_idx,user_id,positive_item_idx,positive_parent_asin,positive_source_category,negative_item_idxs,num_negatives
0,test,0,AE22236AFRRSMQIKGG7TPTB75QEA,8928,B005CG2AX2,Electronics,"[18691, 13759, 9605, 14678, 8039, 2231, 9010, 17890, 18840, 16705, 17790, 8055, 15263, 12509, 264, 10849, 10420, 7318, 2622, 16349, 1528...",100
1,test,1,AE222H3FGXWLHRFUMGMS2RR57NDQ,16006,B00CS43I4Q,Electronics,"[15129, 5229, 18294, 408, 15321, 3885, 2008, 15745, 9085, 265, 7929, 11658, 6705, 2658, 8297, 16577, 6873, 9770, 2293, 6050, 15938, 8651...",100
2,test,2,AE223LQXDVA2DHNB7XKAPILBS2DQ,2544,B0012XFDWO,Electronics,"[16001, 15975, 6181, 1812, 15869, 407, 3495, 184, 6100, 17506, 16725, 3439, 13073, 4070, 11568, 2379, 16909, 18953, 5712, 13843, 2950, 1...",100
3,test,3,AE223NW3QAU2KTAY5MAEU6VPBAUA,6242,B003UE52ME,Electronics,"[14242, 7668, 3212, 10985, 16891, 3001, 8286, 9928, 13934, 5748, 9741, 11678, 17545, 2043, 3236, 6860, 6458, 11047, 12234, 10189, 2504, ...",100
4,test,4,AE223SBXB2PFTMOITN4V53TLSFSA,18557,B00I5LMSE2,Video Games,"[17479, 2856, 18433, 4971, 1444, 16367, 16837, 4471, 15072, 15855, 8550, 6070, 4853, 18049, 14358, 7399, 3251, 11563, 12668, 1038, 16431...",100


## 12 salvataggio degli output

In [23]:
TRAIN_PATH=OUTPUT_DIR /"train.parquet"  #percorso train set
VALIDATION_PATH=OUTPUT_DIR/"validation.parquet" #percorso validation set originale
TEST_PATH=OUTPUT_DIR / "test.parquet"  #percorso test set originale
VALIDATION_EVAL_PATH=OUTPUT_DIR/"validation_eval.parquet" #percorso validation valutabile
TEST_EVAL_PATH=OUTPUT_DIR /"test_eval.parquet" #percorso test valutabile
ITEMS_OUT_PATH=OUTPUT_DIR /"items.parquet" #percorso prodotti finali
QA_OUT_PATH=OUTPUT_DIR / "qa.parquet"  #percorso QA finale
USER_MAPPING_PATH=OUTPUT_DIR /"user_mapping.parquet"  #percorso mapping utenti
ITEM_MAPPING_PATH=OUTPUT_DIR /"item_mapping.parquet"  #percorso mapping prodotti
VALIDATION_NEGATIVE_PATH=OUTPUT_DIR / "negative_samples_validation.parquet"  #percorso negativi validation
TEST_NEGATIVE_PATH=OUTPUT_DIR / "negative_samples_test.parquet" #percorso negativi test
SPLIT_SUMMARY_PATH=OUTPUT_DIR /"split_summary.json" #percorso riepilogo finale
FILTER_HISTORY_PATH=OUTPUT_DIR /"positive_filter_history.csv" #percorso storia del filtro

In [24]:
train_df.to_parquet(TRAIN_PATH,index=False)#salva train set
validation_df.to_parquet(VALIDATION_PATH,index=False) #salva validation set originale
test_df.to_parquet(TEST_PATH,index=False) #salva test set originale
validation_eval_df.to_parquet(VALIDATION_EVAL_PATH,index=False)  #salva validation valutabile
test_eval_df.to_parquet(TEST_EVAL_PATH,index=False)  #salva test valutabile
final_items_df.to_parquet(ITEMS_OUT_PATH,index=False)  #salva prodotti finali
final_qa_df.to_parquet(QA_OUT_PATH,index=False) #salva QA finale
user_mapping_df.to_parquet(USER_MAPPING_PATH,index=False)#salva mapping utenti
item_mapping_df.to_parquet(ITEM_MAPPING_PATH,index=False)#salva mapping prodotti
filter_history_df.to_csv(FILTER_HISTORY_PATH,index=False)#salva storia del filtro positivo
validation_negative_samples_df.to_parquet(VALIDATION_NEGATIVE_PATH,index=False)
test_negatives_samples_df.to_parquet(TEST_NEGATIVE_PATH,index=False)


In [25]:
def density_percentage(num_rows,num_users,num_items):
    possible_pairs=num_users*num_items #calcola numero coppie utente-prodotto possibili
    if possible_pairs==0:
        return None 
    return float(num_rows/possible_pairs*100)

In [26]:
split_summary={  #crea riepilogo finale da salvare
    "category":CATEGORY,
    "dataset_name":DATASET_NAME,
    "source_config": SELECTED_CONFIG,  #salva configurazione di partenza del Notebook 02
    "positive_rating_threshold":int(POSITIVE_RATING_THRESHOLDS),  #salva soglia rating positivo
    "min_user_interactions_after":int(MIN_USER_INTERACTIONS), #salva soglia utente dopo target
    "min_item_interactions_after": int(MIN_ITEM_INTERACTIONS),#salva soglia prodotto dopo target
    "n_negatives": int(N_NEG), #salva numero negativi
    "negative_sampling_mode":NEGATIVE_SAMPLING_MODE,
    "same_category_negative_ratio":float(SAME_CATEGORY_NEGATIVE_RATIO),
    "random_state": int(RANDOM_STATE),  #salva seed
    "temporal_column": TIME_COL,  #salva la colonna temporale usata per lo split
    "original_candidate_interactions": int(len(interactions_df)),#salva interazioni candidate iniziali
    "positive_interactions_before_refilter": int(len(positive_interactions_df)),  #salva positive prima del rifiltro
    "final_interactions": int(len(indexed_interactions_df)),  #salva interazioni finali
    "final_users":int(indexed_interactions_df["user_idx"].nunique()),
    "final_items":int(indexed_interactions_df["item_idx"].nunique()),
    "final_density_pct": density_percentage(len(indexed_interactions_df), indexed_interactions_df["user_idx"].nunique(), indexed_interactions_df["item_idx"].nunique()),
    "train_rows":int(len(train_df)),
    "validation_rows": int(len(validation_df)),
    "test_rows":int(len(test_df)),
    "validation_eval_rows": int(len(validation_eval_df)),#salva righe validation valutabili
    "test_eval_rows": int(len(test_eval_df)),#salva righe test valutabili
    "validation_cold_items": int(len(validation_cold_items)),  #salva item cold-start validation
    "test_cold_items": int(len(test_cold_items)),  # salva item cold-start test
    "category_stats":final_category_stats_df.to_dict(orient="records"),
    "train_path": str(TRAIN_PATH),  # salva percorso train
    "validation_eval_path": str(VALIDATION_EVAL_PATH),  #salva percorso validation eval
    "test_eval_path": str(TEST_EVAL_PATH),  #salva percorso test eval
    "negative_samples_validation_path": str(VALIDATION_NEGATIVE_PATH),  #salva percorso negativi validation
    "negative_samples_test_path": str(TEST_NEGATIVE_PATH),  #salva percorso negativi test
}  # chiude dizionario riepilogo.

SPLIT_SUMMARY_PATH.write_text(json.dumps(split_summary,indent=2,ensure_ascii=False),encoding="utf-8")  #salva riepilogo JSON
display(pd.DataFrame.from_dict(split_summary,orient="index",columns=["value"]))

,value
category,AmazonLike_v1
dataset_name,test_dataset2
source_config,user3_item8
positive_rating_threshold,4
min_user_interactions_after,3
min_item_interactions_after,8
n_negatives,100
negative_sampling_mode,mixed_category
same_category_negative_ratio,0.500000
random_state,42
